# 05 — Test risk-engine (résilience au restart container)

Smoke test du container `fxvol-risk` — étape 5/5 (dernier de la série). Valide que **l'engine récupère proprement après un `docker restart`** : reconnexion IB, sync historique sans crash tzdata, lecture spot+surface, premier nouveau heartbeat en < 30s, greeks reprennent leur publication.

**Pourquoi c'est critique** : risk-engine est mis à jour de temps en temps (deploy nouvelle image, OOM kill, etc.). Le pipeline doit reprendre sans intervention humaine. Côté frontend, un trou de plus de 30s sans greeks signifie que `useRiskStream` voit un freeze visible — le panel risk affiche des valeurs stale.

**Particularité vs market-data 05** : la sync IB de risk-engine est **plus lourde** (reçoit `position`, `updatePortfolio`, `execDetails` × N positions/orders historiques) — ~2-5s avant que les cycles commencent vraiment. Le test tolère donc un peu plus de latence sur le first-heartbeat.

**Couvre** :
1. Seed surface (sinon cycle skip post-restart aussi)
2. Baseline — heartbeat actuel existe (point de départ)
3. Restart container `fxvol-risk` (subprocess `docker restart`)
4. Attendre nouveau heartbeat ≠ baseline en < 30s
5. Confirmer que pubsub `risk_update` reprend : SUBSCRIBE, ≥ 2 messages en 5s
6. État final : heartbeat frais (< 5s), `latest_greeks:portfolio` présent

**Préreq** :
- Notebooks 01-04 verts
- ib-gateway healthy + market-data en cours (alimente `latest_spot`)
- Patch tzdata présent dans l'image (sinon l'IB sync va re-crasher post-restart, cf. fix R9 sandbox)

## Setup + seed surface

In [1]:
import json
import subprocess
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
CONTAINER = "fxvol-risk"
SYMBOL = "EURUSD"

RECOVERY_DEADLINE_S = 30.0
POST_RESTART_LISTEN_S = 6.0  # un peu plus large que market-data (cycle 2s vs 100ms)
MIN_POST_RESTART_MESSAGES = 2

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL")

# Seed surface AVANT le restart pour qu'au reboot, l'engine ait tout pour cycler.
fake_surface = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "surface": {"1M": {"atm": {"iv": 0.07, "strike": 1.17}}},
}
r.set(f"latest_vol_surface:{SYMBOL}", json.dumps(fake_surface), ex=600)
print(f"Connected -> {REDIS_URL}, surface seedé (TTL 600s)")

Connected -> redis://localhost:6380/0, surface seedé (TTL 600s)


## 1. Baseline — heartbeat avant restart

Si pas de baseline → engine déjà cassé, restart va pas réparer. Vérifier notebooks 01/02 d'abord.

In [2]:
print("== 1. baseline heartbeat ==")

baseline_hb = r.get("heartbeat:risk_engine")
record("heartbeat baseline présent",
       baseline_hb is not None,
       baseline_hb if baseline_hb else "<missing>")

if baseline_hb is None:
    raise RuntimeError(
        "Pas de heartbeat baseline — risk-engine non opérationnel.\n"
        "Re-vérifier notebooks 01/02 et le patch _read_spot."
    )

== 1. baseline heartbeat ==
  [OK  ] heartbeat baseline présent  | 2026-04-28T15:29:10.442955Z


## 2. Restart container

On utilise `docker restart fxvol-risk` (commande runtime) plutôt que `docker compose restart risk-engine` qui re-évalue le compose entier (et planterait sans `load_secrets.ps1` à cause du `${IB_PASSWORD:?...}` du service ib-gateway). Cf. notebook 05 du market-data pour le même pattern.

In [3]:
print("== 2. docker restart fxvol-risk ==")

t0 = time.perf_counter()
out = subprocess.run(["docker", "restart", CONTAINER],
                     capture_output=True, text=True)
elapsed = time.perf_counter() - t0

record("docker restart",
       out.returncode == 0,
       f"exit={out.returncode}, took {elapsed:.1f}s")

== 2. docker restart fxvol-risk ==
  [OK  ] docker restart  | exit=0, took 0.6s


## 3. Nouveau heartbeat ≠ baseline en < 30s

**Note** : risk-engine doit reconnecter IB (~2-3s) + re-sync historique du compte (positions, orders, execDetails — ~1-2s) AVANT que les cycles 2s commencent. Donc le first-heartbeat post-restart arrive typiquement à 5-8s du restart.

**Sub-test caché** : si le nouveau heartbeat n'arrive jamais → soit le container ne redémarre pas, soit `tzdata` manque dans l'image (l'IB sync re-crashe en boucle silencieusement) → vérifier `requirements/ib.txt` contient bien `tzdata>=2024.1`.

In [4]:
print(f"== 3. attendre nouveau heartbeat ≤ {RECOVERY_DEADLINE_S}s ==")

t0 = time.perf_counter()
deadline = t0 + RECOVERY_DEADLINE_S
new_hb = None
while time.perf_counter() < deadline:
    current = r.get("heartbeat:risk_engine")
    if current and current != baseline_hb:
        try:
            datetime.fromisoformat(current.replace("Z", "+00:00"))
            new_hb = current
            break
        except ValueError:
            pass
    time.sleep(0.5)

recovery_s = time.perf_counter() - t0
record(f"nouveau heartbeat ≠ baseline en ≤ {RECOVERY_DEADLINE_S}s",
       new_hb is not None,
       f"recovered in {recovery_s:.1f}s — new = {new_hb!r}" if new_hb
       else f"timeout après {recovery_s:.1f}s")

== 3. attendre nouveau heartbeat ≤ 30.0s ==
  [OK  ] nouveau heartbeat ≠ baseline en ≤ 30.0s  | recovered in 1.5s — new = '2026-04-28T15:29:12.643667Z'


## 4. Pub/sub `risk_update` reprend

SUBSCRIBE après le restart, ≥ 2 messages en 6s. Plancher pareil au notebook 03 (cycle 2s).

In [5]:
print(f"== 4. pubsub risk_update — listen {POST_RESTART_LISTEN_S}s ==")

sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
sub.subscribe("risk_update")

messages = []
deadline = time.perf_counter() + POST_RESTART_LISTEN_S
while time.perf_counter() < deadline:
    msg = sub.get_message(timeout=0.1)
    if msg and msg.get("type") == "message":
        try:
            json.loads(msg["data"])
            messages.append(msg["data"])
        except json.JSONDecodeError:
            pass

sub.unsubscribe("risk_update")
sub.close()

record(f"≥ {MIN_POST_RESTART_MESSAGES} messages JSON post-restart",
       len(messages) >= MIN_POST_RESTART_MESSAGES,
       f"{len(messages)} messages")

== 4. pubsub risk_update — listen 6.0s ==
  [OK  ] ≥ 2 messages JSON post-restart  | 3 messages


## 5. État final

In [6]:
print("== 5. état final ==")

current_hb = r.get("heartbeat:risk_engine")
if current_hb:
    try:
        hb_dt = datetime.fromisoformat(current_hb.replace("Z", "+00:00"))
        age_s = (datetime.now(timezone.utc) - hb_dt).total_seconds()
        record("heartbeat frais (age < 5s)",
               age_s < 5.0,
               f"age = {age_s:.2f}s")
    except ValueError as e:
        record("heartbeat parsable", False, str(e))
else:
    record("heartbeat présent", False, "<missing>")

greeks_raw = r.get("latest_greeks:portfolio")
record("latest_greeks:portfolio présent",
       greeks_raw is not None,
       "<missing>" if greeks_raw is None else f"{len(greeks_raw)} bytes JSON")

== 5. état final ==
  [OK  ] heartbeat frais (age < 5s)  | age = 0.25s
  [OK  ] latest_greeks:portfolio présent  | 129 bytes JSON


## Récap final

In [7]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — risk-engine récupère proprement d'un restart.")
    print("Surface risk-engine complètement validée (notebooks 01-05).")
    print("Reste : vol-engine, db-writer, frontend.")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
heartbeat baseline présent                              OK      2026-04-28T15:29:10.442955Z
docker restart                                          OK      exit=0, took 0.6s
nouveau heartbeat ≠ baseline en ≤ 30.0s                 OK      recovered in 1.5s — new = '2026-04-28T15:29:12.643667Z'
≥ 2 messages JSON post-restart                          OK      3 messages
heartbeat frais (age < 5s)                              OK      age = 0.25s
latest_greeks:portfolio présent                         OK      129 bytes JSON
--------------------------------------------------------------------------------------------------------------

6 OK / 0 FAIL  (6 total)

OK — risk-engine récupère proprement d'un restart.
Surface risk-engine complètement validée (notebooks 01-05).
Reste : vol-engine, db-writer, frontend.


## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| baseline absent au §1 | risk-engine pas opérationnel pré-restart | revoir notebooks 01/02 ; vérifier seed surface |
| restart prend > 15s au §2 | signal handler SIGTERM cassé | investiguer `src/services/risk/main.py` |
| §3 timeout 30s | engine ne reprend pas — soit IB connection foire (TrustedIPs ?), soit tzdata manque (sync IB crashe) | `docker logs fxvol-risk --tail 30` ; chercher `ZoneInfoNotFoundError` ou `TimeoutError` |
| §3 OK mais §4 0 messages | bridge api restart pas synchrone, ou cycle skip surface absente | re-seeder surface (TTL 600s peut avoir expiré entre 2 runs longs) |
| §5 heartbeat age > 5s | crashloop : engine reboot, 1 push, re-crash | `docker logs --tail 100` ; chercher exceptions |